# 01 — Grounding: schema graphs and HN-SupCon

Train the embedding model and export column embeddings with database-scoped metadata. PK–FK schema graphs and Node-Weighted Steiner Tree linking remain part of Grounding. Inputs: train questions with `question`, `db_id`, `SQL`; held-out retrieval questions; SQLite databases. Each dataset must be licensed and independently split. Gold SQL is used only for training/retrieval labels.


In [ ]:
import os
import sys
from pathlib import Path

# Attach a repository snapshot as a Kaggle Dataset, or set KAGGLE_REPO_DIR.
_candidates = [Path(os.environ.get("KAGGLE_REPO_DIR", ".")), Path.cwd(), Path.cwd().parent]
_candidates += list(Path("/kaggle/input").glob("*/Optimize-Mars-Sl")) if Path("/kaggle/input").exists() else []
REPO_DIR = next((p.resolve() for p in _candidates if (p / "src").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Set KAGGLE_REPO_DIR to the attached repository directory containing src/")
sys.path.insert(0, str(REPO_DIR))
from src.training.kaggle import KagglePaths, install_dependencies, environment_report
install_dependencies(REPO_DIR)  # Only runs pip when INSTALL_DEPS=1.
paths = KagglePaths.from_env()
SMOKE = paths.smoke
print({"mode": "smoke (NOT a benchmark)" if SMOKE else "train", "data": str(paths.data_dir), "device": paths.device})


In [ ]:
PHASE = 'grounding'
import json
OUTPUT_DIR = paths.output(PHASE)
environment = environment_report(OUTPUT_DIR)
MAX_STEPS = int(os.environ.get("MAX_STEPS", "100"))
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "512"))
SEED = int(os.environ.get("SEED", "42"))
SAVE_STEPS = int(os.environ.get("SAVE_STEPS", "25"))
RESUME = os.environ.get("RESUME_FROM_CHECKPOINT") or os.environ.get("CHECKPOINT_RESUME_SOURCE", "none")
print({"output": OUTPUT_DIR, "resume_from_checkpoint": RESUME, "gpus": environment["gpus"]})

from src.training.kaggle import initialize_checkpoint_backend
checkpoint_backend = initialize_checkpoint_backend(OUTPUT_DIR, PHASE, smoke=SMOKE)


Configure `TRAIN_DATA`, `RETRIEVAL_EVAL_DATA`, `DB_DIR`, and optionally `CARDS_PATH`. Real mode defaults to `train.jsonl`, `retrieval_dev.jsonl`, `databases/` beneath `KAGGLE_DATA_DIR`. Column cards and schema graphs are extracted into output, preserving input files. The embedding checkpoint and index can be trained on an independent account.

In [ ]:
from src.training.runtime import load_records, resolve_db
from src.grounding.column_card import ColumnCardExtractor
from src.grounding.graph_builder import build_graph_from_sqlite
from src.grounding.steiner_tree import extract_compact_subschema
from src.training.train_embedding import run_dry_run, run_embedding_training

TRAIN_DATA = paths.input("TRAIN_DATA", "mini_bird.jsonl" if SMOKE else "train.jsonl")
EVAL_DATA = paths.input("RETRIEVAL_EVAL_DATA", "mini_bird.jsonl" if SMOKE else "retrieval_dev.jsonl")
DB_DIR = paths.input("DB_DIR", "." if SMOKE else "databases")
CARDS_PATH = os.environ.get("CARDS_PATH", str(Path(OUTPUT_DIR) / "column_cards.jsonl"))
if "CARDS_PATH" not in os.environ:
    cards, graphs = [], {}
    for db_id in sorted({r["db_id"] for r in load_records(TRAIN_DATA) + load_records(EVAL_DATA)}):
        db_path = resolve_db(DB_DIR, db_id)
        graph = build_graph_from_sqlite(db_path, db_id)
        for card in ColumnCardExtractor().extract_from_sqlite(db_path):
            cards.append(dict(card.model_dump(), db_id=db_id))
        graphs[db_id] = {"tables": graph.get_nodes(), "edges": [list(f.as_tuple()) for f in graph.foreign_keys]}
    Path(CARDS_PATH).write_text("".join(json.dumps(c) + "\n" for c in cards))
    (Path(OUTPUT_DIR) / "schema_graphs.json").write_text(json.dumps(graphs, indent=2))
if SMOKE:
    summary = dict(run_dry_run(cards_path=CARDS_PATH, bird_path=TRAIN_DATA, output_dir=OUTPUT_DIR), mode="smoke", trained=False)
else:
    summary = run_embedding_training(cards_path=CARDS_PATH, bird_path=TRAIN_DATA,
        eval_path=EVAL_DATA, db_dir=DB_DIR, output_dir=OUTPUT_DIR, dry_run=False,
        model_name_or_path=os.environ.get("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B"),
        device=paths.device, max_steps=MAX_STEPS, max_length=MAX_LENGTH, save_steps=SAVE_STEPS,
        seed=SEED, resume_from_checkpoint=RESUME)
print(json.dumps(summary, indent=2))


## Đồng bộ checkpoint và khôi phục

Trước setup, cấu hình repo HF **private**, Secret `HF_TOKEN`, `CHECKPOINT_SYNC_ENABLED=1`,
`CHECKPOINT_HF_REPO_ID`, `CHECKPOINT_EXPERIMENT_ID` riêng cho mỗi thí nghiệm và
`CHECKPOINT_RESUME_SOURCE=auto`. Xem hướng dẫn tiếng Việt trong `KAGGLE.md`.
Setup kiểm tra quyền riêng tư và in step local/remote đã xác minh trước khi tải model.
`SAVE_STEPS` điều khiển lưu local; `CHECKPOINT_SYNC_EVERY_N_SAVES` điều khiển upload đồng bộ.
Checkpoint cuối được upload khi runner thành công; kiểm tra `checkpoint_sync.jsonl` và cảnh báo backup thất bại.
`none` bắt đầu mới; `local`, `remote`, `auto`, hoặc `RESUME_FROM_CHECKPOINT` là đường dẫn chính xác.
Giữ nguyên dữ liệu/cấu hình khi resume vào output mới. Smoke không kết nối Hub.

Có thể dùng **Save Version → Save & Run All** và đóng trình duyệt. `/kaggle/working` là tạm thời;
chỉ checkpoint đã upload hoàn tất mới có thể phục hồi. Các optimizer step chưa lưu/upload có thể mất.
Không xóa checkpoint tự động. Notebook 05 chỉ đọc artifacts; không đồng bộ ngược checkpoint.
Chưa xác minh Kaggle GPU, NF4 hoặc upload Hub thực tế trong lần triển khai này.
